In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import numpy as np
from sklearn.metrics import mean_squared_error,r2_score
dataset = pd.read_csv('insurance.csv')
dataset.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


ИМПОРТИРУЮ ВСЕ НЕОБХОДИМЫЕ БИБЛИОТЕКИ ЧИТАЮ ФАЙЛ СКАЧАННЫЙ С kaglle вывожу данные чтобы проанализировать таблицы

In [2]:
features = ['age','bmi','children']
X = dataset[features]
y = dataset['charges']


выделяю данные которые не имеют тектового представления выделяю их в отдельный список и формирую X и Y

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, train_size=0.8, random_state=42)

linreg = LinearRegression()
fit_linreg = linreg.fit(X_train, y_train)


делю данные на тестовые и тренировочные
создаю модель линейной регресии и обучаю ее на тренировочных данных


In [4]:
y_predict = fit_linreg.predict(X_test)

y_based = np.full(len(y_test),y_train.mean())


print(mean_squared_error(y_test, y_predict)**0.5)
print(r2_score(y_test, y_predict))



11454.315153980095
0.15489592484270753


получаю результаты работы модели в переменой y_predict

формирую переменую y_based для проверк модели и на базовых вычислениях

вывожу mse и r2 на основе тестовых данных

In [5]:
print(mean_squared_error(y_test, y_based)**0.5)
print(r2_score(y_test, y_based))

12465.610441715768
-0.0009192486886582252


вывожу mse и r2 на основе based данных

модель обученная на тренировочных данных показывает результат лучше чем на базовых вычислениях

RMSE модели около 11454
RMSE baseline модели около 12465
 RMSE так как взят корень из MSE для лучшего понимания
R2 модели около 0.15  это всего 15% разброса честно говоря это не самый лучший результат

In [6]:
dict_int  = {'yes' : 1, 'no' : 0}

dataset['smoker_int'] = dataset['smoker'].map(dict_int)

представляю столбец smoker в числовом виде где yes = 1 no = 0

In [7]:
features1 = ['age','bmi','children','smoker_int']
X1 = dataset[features1]
y1 = dataset['charges']

выделяю данные которые не имеют тектового представления и добавленные данные smoker в числовом виде выделяю их в отдельный список и формирую X и Y

In [8]:
X_train1, X_test1, y_train1, y_test1 = train_test_split(X1,y1, test_size=0.2, train_size=0.8, random_state=42)

linreg1 = LinearRegression()
fit_linreg1 = linreg1.fit(X_train1, y_train1)

делю данные на тестовые и тренировочные
создаю модель линейной регресии и обучаю ее на тренировочных данных


In [9]:
y_predict1 = fit_linreg1.predict(X_test1)

y_based1 = np.full(len(y_test1),y_train1.mean())


print(mean_squared_error(y_test1, y_predict1)**0.5)
print(r2_score(y_test1, y_predict1))


5829.378521780666
0.7811147722517887


получаю результаты работы новой модели в переменой y_predict1 

формирую переменую y_based1 для проверк модели и на базовых вычислениях

вывожу mse и r2 на основе новых тестовых данных

In [10]:
print(mean_squared_error(y_test1, y_based1)**0.5)
print(r2_score(y_test1, y_based1))

12465.610441715768
-0.0009192486886582252


In [11]:
print(linreg1.coef_)
print(linreg.coef_)

print(pd.Series(linreg1.coef_, features1))
print(pd.Series(linreg.coef_, features))

[  257.07115795   327.52263109   427.19997095 23653.21164629]
[220.75776541 332.20247584 563.01943242]
age             257.071158
bmi             327.522631
children        427.199971
smoker_int    23653.211646
dtype: float64
age         220.757765
bmi         332.202476
children    563.019432
dtype: float64


## Вывод по коэффициентам

Коэффициенты модели (при прочих равных признаках):
- `age` = 257: каждый лишний год в среднем добавляет около 257 $ к выплатам;
- `bmi` = 328: каждая единица индекса массы тела добавляет около 328 $;
- `children` = 427: каждый ребёнок добавляет около 427 $;
- `smoker_int` = 23 653: у курящих выплаты в среднем **на 23,7 тысяч $ выше**, чем у некурящих.

Коэффициент показывает прибавку к предсказанию, а не множитель. Напрямую сравнивать коэффициенты разных признаков нельзя, потому что у них разный масштаб: `smoker_int` меняется всего от 0 до 1, а `age` на десятки лет. Если сравнивать влияние на всём диапазоне значений (возраст от 18 до 64 лет, то есть около 46 лет × 257 ≈ 11,8 тысяч $), то курение (около 23,7 тысяч $) всё равно влияет примерно вдвое сильнее возраста и является самым значимым признаком.

При добавлении `smoker` коэффициенты остальных признаков изменились (например, у `children` с 563 до 427, у `age` с 221 до 257): модель перераспределила вклад между признаками, часть информации раньше «пряталась» за другими столбцами.

Коэффициенты описывают связи в данных, а не причины: по ним нельзя утверждать, что именно курение «вызывает» рост выплат, а можно говорить лишь о том, что в этих данных у курящих выплаты выше.

In [12]:
y_train_pred1 = linreg1.predict(X_train1)
print(mean_squared_error(y_train1, y_train_pred1)**0.5)
print(r2_score(y_train1, y_train_pred1))


6113.067842369202
0.7410887501824264


метрики test и train отчилются не сильно у test (rmse 5800 r2 0.78) у train (rmse 6100 r2 0.74) это говорит о том что модель не переобучена

Вывожу коэфициенты по названиям признаков

Выплаты курящим у smoker intв  среднем выше примерно на 23,7 тысячи долларов, чем у некурящего

модель обученная на новых тренировочных данных показывает результат намного лучше чем прошлая модель без числового представления smoker

RMSE модели около 5830
RMSE baseline модели около 12465
 RMSE так как взят корень из MSE для лучшего понимания
R2 модели около 0.78  это уже 78% разброса что намного лучше прошлого результата равного 15 % это изменение произошло из за того что я что добавил признак, который несёт основную информацию о выплатах

Модель является переоученной когда она показывает околоидеальный результат на тестовых данных так как была переучана на них а когда поступают неизвестные ранее данные она показывает плохой результат

недообученая модель это когда она не учитывает некоторые признаки или не сопоставляет их правильно